In [27]:
%run ../tardis_eda.ipynb

# Tardis map : initialisation

- modules calls
- define exit values

In [ ]:
import pandas as pd
import folium as flm
from datetime import date
from tqdm import trange
from os import getcwd, chdir

cwd = getcwd()
if cwd.endswith("/bonus"):
    chdir(cwd + "/..")


EPITECH_SUCCESS = 0
EPITECH_FAILURE = 84

GET_NBR_ROWS = lambda df: df.shape[0]
MONTH_IN_YEAR = 12
MIN_IN_H = 60
SEC_IN_MIN = 60
SEC_IN_H = MIN_IN_H * SEC_IN_MIN
H_IN_DAY = 24

MAX_DATA_AGE = 36

# MAP_CENTER = (48.836536, 2.35632)
MAP_CENTER = (46.92475, 2.51240)
MAP_ZOOM = 6.2

COLOR_STEP = 5
STATION_ICON = "train"
LINE_WEIGHT = 2.5

#### Map creation

no argument

return the folium Map object centered on `MAP_CENTER` and zoom at `MAP_ZOOM`

In [29]:
def init_map() -> flm.Map:
    map = flm.Map(location=MAP_CENTER, zoom_start=MAP_ZOOM, tiles="OpenStreetMap")
    tl = flm.TileLayer(
        tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
        attr="&copy; <a href='https://www.openstreetmap.org/copyright'>OpenStreetMap</a> contributors",
    )
    tl.options["Referrer-Policy"] = "no-referrer-when-downgrade"
    tl.add_to(map)
    return map

In [30]:
def get_field_value(ser: pd.Series, wanted_type) -> any:
    val = 0
    # val = ser.values[0]
    for e in ser:
        val = e
        break
    if wanted_type == date and type(val) == pd.Timestamp:
        val = val.date()
    pd_type = wanted_type
    if wanted_type == int:
        pd_type = float
    if type(val) == pd_type:
        if wanted_type == int:
            return int(val)
        return val
    if wanted_type == str:
        return ""
    return None

#### Get the stations names

arguments :
- csv (panda DataFrame), the dataframe

return the list of stations names (list of str)\
return None in case of error

In [31]:
def get_stations_data() -> dict[str : tuple[float, float]]:
    try:
        csv = pd.read_csv("bonus/stations_coords.csv", sep=",")
    except:
        return None
    for c in ("Latitude", "Longitude"):
        csv[c] = pd.to_numeric(csv[c], errors="coerce")
    data = {}
    for i in range(GET_NBR_ROWS(csv)):
        name = get_field_value(csv.iloc[[i]]["Name"], str)
        lat = get_field_value(csv.iloc[[i]]["Latitude"], float)
        long = get_field_value(csv.iloc[[i]]["Longitude"], float)
        if (
            type(name) == str
            and type(lat) in (int, float)
            and type(long) in (int, float)
        ):
            data[name] = (lat, long)
    return data

#### Check if a row data is enough recent

arguments :
- row (panda Series), the row

return True if the row data is at most `MAX_DATA_AGE` months old
else or in case of error, return False

In [32]:
def is_enough_recent(row: pd.Series) -> bool:
    date_to_months = lambda d: d.year * MONTH_IN_YEAR + d.month
    today = date_to_months(date.today())
    d = get_field_value(row["Date"], date)
    if type(d) != date:
        return False
    m = date_to_months(d)
    return abs(today - m) <= MAX_DATA_AGE

#### Get the stations weight for the map

arguments :
- csv (panda DataFrame), the dataframe
- station (str), the station name

return the mean of "Average delay of all trains at departure" counting only rows at most `MAX_DATA_AGE` months old and whose departure station is the given `station`\
return -1 if no data match the given `station`

In [33]:
def get_station_dep_delay(csv: pd.DataFrame, station: str) -> float:
    mean = 0
    n = 0
    today = date.today().year * MONTH_IN_YEAR + date.today().month
    rows = csv.query(
        f"`Year` * {MONTH_IN_YEAR} + `Month` <= {
            today - MAX_DATA_AGE
        } & `Departure station` == '{station.upper()}'"
    )

    for i in range(GET_NBR_ROWS(rows)):
        tmp = get_field_value(
            rows.iloc[[i]]["Average delay of all trains at departure"], float
        )
        if type(tmp) not in (int, float):
            continue
        n += 1
        mean += tmp
    if n <= 0:
        return -1
    return mean / n

#### Get the vertices weight for the map

arguments :
- csv (panda DataFrame), the dataframe
- statiobs (tuple of 2 str), the (station_name1, station_name2) tuple

return the mean of "Average delay of all trains at arrival" counting only rows at most `MAX_DATA_AGE` months old and whose departure and arrival stations match the given `stations` (order has no importance)\
return -1 if no data match the given `stations` or if the both stations are equal

In [34]:
def check_line(row: pd.Series, stations: tuple[str, str]) -> bool:
    s1, s2 = stations
    s1, s2 = s1.upper(), s2.upper()
    dest = get_field_value(row["Departure station"], str)
    arr = get_field_value(row["Arrival station"], str)
    if type(dest) != str or type(arr) != str:
        return False
    return (dest.upper(), arr.upper()) == (s1, s2) or (dest.upper(), arr.upper()) == (
        s2,
        s1,
    )


def get_line_mean_delay(csv: pd.DataFrame, stations: tuple[str, str]) -> float:
    mean = 0
    n = 0
    s1, s2 = stations[0].upper(), stations[1].upper()
    today = date.today().year * MONTH_IN_YEAR + date.today().month
    rows = csv.query(
        f"`Year` * {MONTH_IN_YEAR} + `Month` <= {
            today - MAX_DATA_AGE
        } & (`Departure station` == '{s1}' & `Arrival station` == '{
            s2
        }') | (`Departure station` == '{s2}' & `Arrival station` == '{s1}')"
    )

    if s1 == s2:
        return -1
    for i in range(GET_NBR_ROWS(rows)):
        tmp = get_field_value(
            rows.iloc[[i]]["Average delay of all trains at arrival"], float
        )
        if type(tmp) not in (int, float):
            continue
        n += 1
        mean += tmp
    if n <= 0:
        return -1
    return mean / n

#### Get color for objects + Convertion time (in minutes) to time (formatted as (j) hh:mm:ss)

color guide :
- <span style="color: green">green</span> = average delay <= 5 minutes
- <span style="color: blue">blue</span> = 5 minutes < average delay <= 10 minutes
- <span style="color: yellow">yellow</span> = 10 minutes < average delay <= 15 minutes
- <span style="color: orange">orange</span> = 15 minutes < average delay <= 20 minutes
- <span style="color: red">red</span> = 20 minutes < average delay <= 30 minutes
- <span style="color: purple">purple</span> = average delay > 30 minutes
- <span style="color: gray">gray</span> = no data

In [35]:
def get_color(delay: float) -> str:
    if delay < 0:
        return "gray"
    colors = ("green", "blue", "yellow", "orange", "red", "red", "purple")
    i = 0
    while delay >= COLOR_STEP and i < (len(colors) - 1):
        delay -= COLOR_STEP
        i += 1
    return colors[i]


def min_to_time(nbr: int) -> str:
    if nbr < 0:
        return "-" + min_to_time(-nbr)
    n_sec = int(round(nbr * SEC_IN_MIN, 0))
    d = n_sec // (SEC_IN_H * H_IN_DAY)
    n_sec = n_sec % (SEC_IN_H * H_IN_DAY)
    h = n_sec // SEC_IN_H
    n_sec = n_sec % SEC_IN_H
    m = n_sec // SEC_IN_MIN
    sec = n_sec % SEC_IN_MIN
    text = f"{d}j" if d > 0 else ""
    return text + f"{h:02}:{m:02}:{sec:02}"

#### Draw stations one the map :

arguments :
- map (folium Map), the map object
- csv (panda DataFrame), the dataframe
- stations ({str : (float, float)} dict), the stations data

draw the stations markers on the map\
doesn't return anything

In [ ]:
def draw_stations(
    map: flm.Map, csv: pd.DataFrame, stations: dict[str : tuple[float, float]]
) -> None:
    print("Draw stations")
    names = list(stations.keys())
    for i in trange(0, len(names), 1):
        k = names[i]
        delay = get_station_dep_delay(csv, k)
        c = get_color(delay)
        if c == "yellow":
            c = "beige"
        if delay < 0:
            popup = f"{k} : NaN"
        else:
            popup = f"{k} : {min_to_time(delay)}"
        flm.Marker(
            location=list(stations[k]),
            icon=flm.Icon(color=c, prefix="fa", icon=STATION_ICON),
            popup=flm.Popup(popup, max_width=None),
        ).add_to(map)

#### Draw routes one the map :

arguments :
- map (folium Map), the map object
- csv (panda DataFrame), the dataframe
- stations ({str : (float, float)} dict), the stations data

draw the route polylines on the map\
doesn't return anything

In [37]:
def draw_lines(
    map: flm.Map, csv: pd.DataFrame, stations: dict[str : tuple[float, float]]
) -> None:
    print("Draw lines :")
    names = list(stations.keys())
    l = len(names)
    for i in trange(l - 1):
        k1 = names[i]
        for j in range(i + 1, l):
            k2 = names[j]
            if k2 == k1:
                continue
            delay = get_line_mean_delay(csv, (k1, k2))
            if delay < 0:
                continue
            c = get_color(delay)
            popup = f"{k1} ↔ {k2} : {min_to_time(delay)}"
            flm.PolyLine(
                [list(stations[k1]), list(stations[k2])],
                color=c,
                weight=LINE_WEIGHT,
                tooltip=popup,
            ).add_to(map)
    print()

In [38]:
def draw_map(stations: bool = True, lines: bool = False) -> flm.Map:
    if stations or lines:
        csv = tardis_eda("dataset.csv", separator=";")
        data = get_stations_data()
        if stations is None or csv is None:
            return None
        csv = csv[0]
    map = init_map()
    if stations:
        draw_stations(map, csv, data)
    if lines:
        draw_lines(map, csv, data)
    return map


def main_map() -> int:
    map = draw_map(stations=True, lines=True)
    if map is None:
        chdir(getcwd() + "/bonus")
        return EPITECH_FAILURE
    print("Save the map")
    map.save("maps/map_mean.html")
    print("Finished !")
    chdir(getcwd() + "/bonus")
    return EPITECH_SUCCESS


if __name__ == "__main__":
    main_map()

Draw stations


100%|██████████| 59/59 [00:00<00:00, 64.22it/s] 


Draw lines :


100%|██████████| 58/58 [00:05<00:00, 10.45it/s]



Save the map
Finished !
